### IMPORTING LIBRARIES


In [1]:
# Importing the python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# Loading the dataset
DATA_DIR = os.path.join("..", "Finlora_Dataset")

customer_transaction_data = pd.read_csv(
    os.path.join(DATA_DIR, "FinLora_Customer_Transaction_Dataset.csv")
)
customer_transaction_data.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [3]:
#checking for number of rows and columns
customer_transaction_data.shape

(11400, 26)

In [4]:
#checking for all the columns in the dataset
customer_transaction_data.columns

Index(['transaction_id', 'customer_id', 'timestamp', 'home_country',
       'source_currency', 'dest_currency', 'channel', 'amount_src',
       'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'device_id',
       'new_device', 'ip_address', 'ip_country', 'location_mismatch',
       'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score',
       'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud'],
      dtype='object')

## Missing Value Assessment

Running `isnull().sum()` shows missing data is limited to 7 of the 26 columns; every other column is fully populated across all 11,400 transactions.

| Column | Missing | % of rows |
|---|---|---|
| timestamp | 29 | 0.25% |
| amount_usd | 305 | 2.68% |
| fee | 295 | 2.59% |
| ip_address | 305 | 2.68% |
| ip_country | 301 | 2.64% |
| kyc_tier | 300 | 2.63% |
| device_trust_score | 295 | 2.59% |

**Observations:**
- The largest gap affects under 3% of rows, so no column needs to be dropped outright.
- `timestamp` missingness is small (29 rows) but important, it currently blocks temporal feature engineering for those rows specifically.
- `amount_usd`, `fee`, `ip_address`, `ip_country`, `kyc_tier`, and `device_trust_score` all cluster around 295-305 missing rows. That similarity suggests these gaps may come from the same subset of transactions failing to capture multiple fields together, rather than each column failing independently.

**Next step:** each column will be handled individually during cleaning, dropping rows only where a field is essential and unrecoverable, and imputing elsewhere with a defensible method (e.g. median for numeric fields, an explicit "unknown" category for `kyc_tier`).

In [5]:
# checking for missing values
customer_transaction_data.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [6]:
# checking data types and non-null counts
customer_transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  obj

In [7]:
# viewing the full first row.
customer_transaction_data.iloc[0]

transaction_id               fee8542d-8ee6-4b0d-9671-c294dd08ed26
customer_id                  402cccc9-28de-45b3-9af7-cc5302aa1f93
timestamp                        2022-10-03 18:40:59.468549+00:00
home_country                                                   US
source_currency                                               USD
dest_currency                                                 CAD
channel                                                       ATM
amount_src                                                 278.19
amount_usd                                                 278.19
fee                                                          4.25
exchange_rate_src_to_dest                                1.351351
device_id                    9f292dcc-3297-4947-a260-6a1ef69041ff
new_device                                                  False
ip_address                                         221.78.171.180
ip_country                                                     US
location_m

## Data Dictionary

| Column | Dtype | Description | Example (row 0) |
|---|---|---|---|
| transaction_id | object (String, UUID) | Unique identifier for each transaction | fee8542d-8ee6-4b0d-9671-c294dd08ed26 |
| customer_id | object (String, UUID) | Unique identifier for the customer initiating the transaction | 402cccc9-28de-45b3-9af7-cc5302aa1f93 |
| timestamp | object *(needs conversion to Datetime)* | Date and time the transaction occurred | 2022-10-03 18:40:59.468549+00:00 |
| home_country | object (String) | Country where the customer's account is registered | US |
| source_currency | object (String) | Currency used by the sender | USD |
| dest_currency | object (String) | Currency received by the recipient | CAD |
| channel | object (Categorical) | Platform used for the transaction | ATM |
| amount_src | object *(should be Float — needs cleaning)* | Transaction amount in the source currency | 278.19 |
| amount_usd | float64 | Transaction amount converted to USD | 278.19 |
| fee | float64 | Transaction fee charged by the platform | 4.25 |
| exchange_rate_src_to_dest | float64 | Exchange rate applied from source currency to destination currency | 1.351351 |
| device_id | object (String, UUID) | Unique identifier for the device used | 9f292dcc-3297-4947-a260-6a1ef69041ff |
| new_device | bool | Whether this device is new for the customer | False |
| ip_address | object (String) | IP address used for the transaction | 221.78.171.180 |
| ip_country | object (String) | Country detected from the IP address | US |
| location_mismatch | bool | Whether ip_country differs from home_country | False |
| ip_risk_score | float64 | Risk score assigned to the IP address | 0.123 |
| kyc_tier | object (Categorical — not numeric) | Customer KYC verification level | standard |
| account_age_days | int64 | Number of days since account creation | 263 |
| device_trust_score | float64 | Trust score assigned to the device | 0.522 |
| chargeback_history_count | int64 | Number of previous chargebacks | 0 |
| risk_score_internal | float64 | Finlora's existing internal fraud risk score | 0.223 |
| txn_velocity_1h | int64 | Number of transactions in the past 1 hour | 0 |
| txn_velocity_24h | int64 | Number of transactions in the past 24 hours | 0 |
| corridor_risk | float64 | Risk score of the country-to-country transaction corridor | 0.0 |
| is_fraud | int64 (Binary: 1 = Fraud, 0 = Legitimate) | Fraud label — your model's target variable | 0 |

In [8]:
# isolate exactly which rows fail to convert to a number
non_numeric_amount_src = customer_transaction_data[
    pd.to_numeric(customer_transaction_data['amount_src'], errors='coerce').isna()
]
non_numeric_amount_src[['transaction_id', 'amount_src']]

,transaction_id,amount_src
4812,95df4052-29be-470f-87f9-61e5173370b2,"9,998.85"
8036,5569192e-3bf1-47e3-9723-d63c3dd30cad,"9,995.95"
8878,19a415c4-05b3-40b5-ad6d-e7d34247e050,"1,541.55"
9444,99a53933-5876-47df-ad3e-7976b48c01b4,"9,991.24"


In [9]:
# checking for duplicate rows
customer_transaction_data.duplicated().sum()

np.int64(200)

In [10]:
# checking the class balance of the fraud label
customer_transaction_data['is_fraud'].value_counts(normalize=True) * 100

is_fraud
0    91.254386
1     8.745614
Name: proportion, dtype: float64

In [11]:
# confirm these are true duplicate transaction records, not a coincidence
customer_transaction_data['transaction_id'].duplicated().sum()

np.int64(200)

## Summary/Observations

- The dataset contains 11,400 rows and 26 columns before cleaning.

- `timestamp` is stored as an object instead of datetime, and `amount_src` is stored as an object instead of a numeric type, both need correcting before analysis. The `amount_src` issue traces to four rows containing comma-formatted values (e.g. "9,998.85"), three of which sit just under the $10,000 mark, a pattern worth revisiting during fraud analysis as a possible sign of structuring.

- Seven columns contain missing values: `timestamp` (29), `amount_usd` (305), `fee` (295), `ip_address` (305), `ip_country` (301), `kyc_tier` (300), and `device_trust_score` (295). The largest gap is under 3% of rows, so no column needs to be dropped outright.

- 200 fully duplicate transaction records were found and confirmed at the `transaction_id` level as well, meaning these are genuine repeated records, not a coincidence, and are safe to remove.

- The target variable `is_fraud` is highly imbalanced: 91.25% legitimate vs. 8.75% fraudulent. This will need to be addressed during modelling through class weighting or resampling, since accuracy alone would be a misleading metric on data this skewed.

These issues are addressed in the Data Cleaning section below.

DATA CLEANING AND PREPARATION

In [12]:
# converting timestamp to datetime
customer_transaction_data['timestamp'] = pd.to_datetime(customer_transaction_data['timestamp'], errors='coerce')

# converting amount_src to numeric (stripping comma formatting first, so real values aren't lost)
customer_transaction_data['amount_src'] = pd.to_numeric(
    customer_transaction_data['amount_src'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

In [13]:
# checking the dataset again to confirm the data types
customer_transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  object             
 1   customer_id                11400 non-null  object             
 2   timestamp                  11339 non-null  datetime64[ns, UTC]
 3   home_country               11400 non-null  object             
 4   source_currency            11400 non-null  object             
 5   dest_currency              11400 non-null  object             
 6   channel                    11400 non-null  object             
 7   amount_src                 11400 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  de

In [14]:
# reload the raw timestamp column to compare against the converted one
raw_timestamps = pd.read_csv(
    os.path.join(DATA_DIR, "FinLora_Customer_Transaction_Dataset.csv"),
    usecols=['timestamp']
)['timestamp']

# rows where a real value existed originally but conversion turned it into NaT
newly_broken = customer_transaction_data.loc[
    customer_transaction_data['timestamp'].isna() & raw_timestamps.notna(),
    ['transaction_id']
].copy()
newly_broken['original_timestamp'] = raw_timestamps[newly_broken.index]
newly_broken

,transaction_id,original_timestamp
40,d36a0593-5a44-4174-a5ed-cc7317bc8524,2025/13/40 25:61:00
210,7f3f5c09-b75c-4355-aca4-d70e169d6df6,0000-00-00T00:00:00Z
1958,3d21deb2-a290-474e-9cd4-664294f0fa69,0000-00-00T00:00:00Z
2224,5487dc9b-4f24-4b76-a0d5-c2f0b94bd155,0000-00-00T00:00:00Z
2462,277ae243-8d24-49e1-9938-185551be6e05,0000-00-00T00:00:00Z
2480,b79f933f-272d-49bf-bf4b-b2b6ed5662ab,0000-00-00T00:00:00Z
2981,f8474028-9a70-4df5-b9f5-87541cf9bf65,0000-00-00T00:00:00Z
3982,545094a5-5d3f-495a-8df0-f697f37bb414,0000-00-00T00:00:00Z
4193,8e26f846-c088-4c10-9410-613e06bc61b1,2025/13/40 25:61:00
4593,b1018d49-8b60-43ba-9e8f-712e16aeacb9,2025/13/40 25:61:00


During data cleaning, the `timestamp` and `amount_src` variables were converted to their correct data types using `pd.to_datetime()` and `pd.to_numeric()`, both with `errors='coerce'`. Unlike a direct conversion, `amount_src` was first stripped of comma formatting — thousands separators found in 4 rows (e.g. `"9,998.85"`), before casting, which preserved all 11,400 values with zero additional missing data, rather than losing those rows to the conversion.

`timestamp` increased from 29 to 61 missing values after conversion. This is not random corruption: all 32 new nulls trace to two identifiable placeholder patterns in the raw data - `"0000-00-00T00:00:00Z"` (21 rows) and `"2025/13/40 25:61:00"` (11 rows, where every date/time component sits exactly one past its valid maximum). Both represent transactions where no real timestamp was ever recorded, rather than a parsing failure, and are correctly treated as missing going forward. This count may drop slightly once duplicate records are removed, since at least one affected row was also a duplicate transaction.

In [15]:
# Rechecking for missing data
customer_transaction_data.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     61
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [16]:
# is the missingness actually tied to channel type, not random?
customer_transaction_data[customer_transaction_data['ip_address'].isnull()]['channel'].value_counts()

channel
mobile       193
web           79
ATM           28
mobille        1
MOBILE         1
WEB            1
unknown        1
 mobile        1
Name: count, dtype: int64

In [17]:
# checking for inconsistent formatting across the whole channel column
customer_transaction_data['channel'].value_counts()

channel
mobile       6366
web          3727
ATM          1008
mobille        60
 mobile        48
MOBILE         47
unknown        37
WEB            36
 web           34
weeb           24
ATm             9
 ATM            4
Name: count, dtype: int64

In [18]:
# correcting genuine misspellings the strip/lower fix can't catch
channel_corrections = {
    'mobille': 'mobile',
    'weeb': 'web',
}
customer_transaction_data['channel'] = customer_transaction_data['channel'].replace(channel_corrections)

In [19]:
# standardizing channel formatting: trim whitespace and lowercase
customer_transaction_data['channel'] = customer_transaction_data['channel'].str.strip().str.lower()

In [20]:
# confirming channel formatting is clean after corrections and strip/lower
customer_transaction_data['channel'].value_counts()

channel
mobile     6521
web        3821
atm        1021
unknown      37
Name: count, dtype: int64

In [21]:
# removing duplicate transaction records
customer_transaction_data = customer_transaction_data.drop_duplicates()
customer_transaction_data.shape

(11200, 26)

## Variables With Missing Values That Need Imputation

- `amount_usd`: 305 missing values, filled by multiplying `amount_src` by the exchange rate for that currency.

- `fee`: 295 missing values, filled by the median fee within each `channel` group, falling back to the overall median where channel-level data isn't available.

- `kyc_tier`: 300 missing values, filled with the most frequent `kyc_tier` (mode).

- `device_trust_score`: 295 missing values, filled by the median within each `kyc_tier` and `new_device` group, falling back to the overall median.

- `ip_country`: 301 missing values, handled conditionally rather than with one blanket rule. Rows where `location_mismatch` was `False` were filled with `home_country`, since no mismatch was recorded. The remaining rows already had `location_mismatch = True`, meaning the IP country was known to differ from `home_country`, so these were filled with a distinct `"unknown"` placeholder instead of a value that would contradict the existing flag.

- `ip_address`: 305 missing values. Rather than dropping these rows, `ip_address` was filled with `"unknown"`. It is a high-cardinality identifier, not a feature the model would use directly, and 7 of these rows are confirmed fraud cases, dropping them would have removed real examples from an already-imbalanced target.

- `timestamp`: 61 missing values were dropped, since no reasonable value can be imputed for a missing date and time, and the volume is very small relative to the full dataset.

- `amount_src`: showed 4 missing values in the initial pass, but these were fully recovered earlier by stripping comma formatting before converting to numeric, so no rows needed to be dropped for this variable.

In [22]:
# filling the missing values of the fee variable
if 'fee' in customer_transaction_data.columns:
    if 'channel' in customer_transaction_data.columns:
        customer_transaction_data['fee'] = customer_transaction_data.groupby('channel')['fee'].transform(lambda y: y.fillna(y.median()))
    customer_transaction_data['fee'] = customer_transaction_data['fee'].fillna(customer_transaction_data['fee'].median())

In [23]:
# filling missing kyc_tier values with the most frequent tier (mode)
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].fillna(
    customer_transaction_data['kyc_tier'].mode()[0]
)

In [24]:
# filling missing device_trust_score values by kyc_tier/new_device group median, falling back to the overall median
customer_transaction_data['device_trust_score'] = customer_transaction_data.groupby(
    ['kyc_tier', 'new_device']
)['device_trust_score'].transform(lambda x: x.fillna(x.median()))
customer_transaction_data['device_trust_score'] = customer_transaction_data['device_trust_score'].fillna(
    customer_transaction_data['device_trust_score'].median()
)

c:\Users\hp\Downloads\Fraudulent_Transaction_Detection_for_Finlora\finlora_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\hp\Downloads\Fraudulent_Transaction_Detection_for_Finlora\finlora_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [25]:
# checking whether ip_address missingness disproportionately affects fraud cases before deciding how to handle it
customer_transaction_data[customer_transaction_data['ip_address'].isnull()]['is_fraud'].value_counts()

is_fraud
0    293
1      7
Name: count, dtype: int64

In [26]:
# checking location_mismatch among the missing ip_country rows before deciding on a fill rule
customer_transaction_data[customer_transaction_data['ip_country'].isnull()]['location_mismatch'].value_counts()

location_mismatch
False    265
True      31
Name: count, dtype: int64

In [27]:
# where no mismatch was flagged, home_country is a safe assumption for the missing ip_country
mask_no_mismatch = customer_transaction_data['ip_country'].isnull() & (customer_transaction_data['location_mismatch'] == False)
customer_transaction_data.loc[mask_no_mismatch, 'ip_country'] = customer_transaction_data.loc[mask_no_mismatch, 'home_country']

# where a mismatch WAS flagged, we know it isn't home_country, but not what it actually is, so use a distinct placeholder
customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].fillna('unknown')

In [28]:
# filling missing ip_address values with a placeholder, since it is a high-cardinality identifier not used directly as a feature
customer_transaction_data['ip_address'] = customer_transaction_data['ip_address'].fillna('unknown')

In [29]:
# dropping the remaining rows with a missing timestamp, since no value can be reasonably imputed
customer_transaction_data = customer_transaction_data.dropna(subset=['timestamp'])
customer_transaction_data.shape

(11140, 26)

In [30]:
# rechecking for missing values after the full cleaning and imputation pass
customer_transaction_data.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                      0
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   300
fee                            0
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                     0
ip_country                     0
location_mismatch              0
ip_risk_score                  0
kyc_tier                       0
account_age_days               0
device_trust_score             0
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [31]:
# calculating exchange rates per currency
exchange_rates = customer_transaction_data[customer_transaction_data['amount_usd'].notna()].groupby('source_currency').apply(
    lambda x: (x['amount_usd'] / x['amount_src']).mean()
).to_dict()
exchange_rates

C:\Users\hp\AppData\Local\Temp\ipykernel_28528\1283422008.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  exchange_rates = customer_transaction_data[customer_transaction_data['amount_usd'].notna()].groupby('source_currency').apply(


{'CAD': 0.7225170084075527,
 'GBP': 1.2241748355637065,
 'USD': 0.9837313041196536}

In [32]:
# filling the missing values of amount_usd using amount_src and the exchange rate
customer_transaction_data['amount_usd'] = customer_transaction_data.apply(
    lambda row: row['amount_usd'] if pd.notna(row['amount_usd']) else row['amount_src'] * exchange_rates.get(row['source_currency'], 1),
    axis=1
)

In [33]:
# confirming no stray channel variants survived the corrections + strip/lower steps
customer_transaction_data['channel'].unique()

array(['atm', 'web', 'mobile', 'unknown'], dtype=object)

## Sanity Checks After Cleaning

Cleaning fixed dtypes, duplicates, categorical formatting, and missing values. This section checks that the cleaned data is actually sensible, not just complete.

- Check for invalid numbers, negative values in columns that should never be negative (monetary amounts, trust scores, velocity counts).
- Check whether any transaction timestamp falls in the future.
- Check that values fall within a plausible range for each variable.
- Inspect for logical inconsistency between related variables (for example, a transaction flagged as high risk with a trust score that suggests otherwise).

In [34]:
# checking negative values in numeric columns that should never be negative
negative_counts = {
    'amount_src': (customer_transaction_data['amount_src'] < 0).sum(),
    'amount_usd': (customer_transaction_data['amount_usd'] < 0).sum(),
    'fee': (customer_transaction_data['fee'] < 0).sum(),
    'device_trust_score': (customer_transaction_data['device_trust_score'] < 0).sum(),
    'txn_velocity_1h': (customer_transaction_data['txn_velocity_1h'] < 0).sum(),
    'txn_velocity_24h': (customer_transaction_data['txn_velocity_24h'] < 0).sum(),
    'risk_score_internal': (customer_transaction_data['risk_score_internal'] < 0).sum(),
}
negative_counts

{'amount_src': np.int64(100),
 'amount_usd': np.int64(3),
 'fee': np.int64(100),
 'device_trust_score': np.int64(200),
 'txn_velocity_1h': np.int64(200),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

In [35]:
# inspecting the actual negative rows before deciding how to handle them
for col in ['amount_src', 'amount_usd', 'fee', 'device_trust_score', 'txn_velocity_1h']:
    print(col)
    print(customer_transaction_data.loc[customer_transaction_data[col] < 0, [col, 'is_fraud']].describe())
    print()

amount_src
        amount_src    is_fraud
count   100.000000  100.000000
mean   -312.655600    0.040000
std     997.888389    0.196946
min   -9997.160000    0.000000
25%    -269.200000    0.000000
50%    -157.850000    0.000000
75%     -83.647500    0.000000
max     -22.570000    1.000000

amount_usd
       amount_usd  is_fraud
count    3.000000       3.0
mean  -187.866446       0.0
std    168.759225       0.0
min   -367.807297       0.0
25%   -265.238553       0.0
50%   -162.669808       0.0
75%    -97.896021       0.0
max    -33.122233       0.0

fee
         fee  is_fraud
count  100.0     100.0
mean    -1.0       0.0
std      0.0       0.0
min     -1.0       0.0
25%     -1.0       0.0
50%     -1.0       0.0
75%     -1.0       0.0
max     -1.0       0.0

device_trust_score
       device_trust_score    is_fraud
count               200.0  200.000000
mean                 -0.1    0.020000
std                   0.0    0.140351
min                  -0.1    0.000000
25%                  -0.

In [36]:
# treating the exact-value negative sentinels as missing again, same pattern as the timestamp sentinels
customer_transaction_data.loc[customer_transaction_data['fee'] == -1, 'fee'] = np.nan
customer_transaction_data.loc[customer_transaction_data['device_trust_score'] == -0.1, 'device_trust_score'] = np.nan
customer_transaction_data.loc[customer_transaction_data['txn_velocity_1h'] == -1, 'txn_velocity_1h'] = np.nan

In [37]:
# confirming the sentinel values are now marked missing
customer_transaction_data[['fee', 'device_trust_score', 'txn_velocity_1h']].isnull().sum()

fee                   100
device_trust_score    200
txn_velocity_1h       200
dtype: int64

In [38]:
# re-imputing fee with the same channel-median logic used earlier
customer_transaction_data['fee'] = customer_transaction_data.groupby('channel')['fee'].transform(lambda y: y.fillna(y.median()))
customer_transaction_data['fee'] = customer_transaction_data['fee'].fillna(customer_transaction_data['fee'].median())

# re-imputing device_trust_score with the same kyc_tier/new_device group logic used earlier
customer_transaction_data['device_trust_score'] = customer_transaction_data.groupby(
    ['kyc_tier', 'new_device']
)['device_trust_score'].transform(lambda x: x.fillna(x.median()))
customer_transaction_data['device_trust_score'] = customer_transaction_data['device_trust_score'].fillna(
    customer_transaction_data['device_trust_score'].median()
)

# txn_velocity_1h had no missing values the first time around, filling with the overall median now
customer_transaction_data['txn_velocity_1h'] = customer_transaction_data['txn_velocity_1h'].fillna(
    customer_transaction_data['txn_velocity_1h'].median()
)

In [39]:
# confirming the sentinel columns are fully re-imputed
customer_transaction_data[['fee', 'device_trust_score', 'txn_velocity_1h']].isnull().sum()

fee                   0
device_trust_score    0
txn_velocity_1h       0
dtype: int64

In [40]:
# amount_src and amount_usd have no refund/transaction-type column to justify a negative value,
# treating this as a sign-flip data entry error and correcting it
customer_transaction_data['amount_src'] = customer_transaction_data['amount_src'].abs()
customer_transaction_data['amount_usd'] = customer_transaction_data['amount_usd'].abs()

In [41]:
# final check confirming no negative values remain in any numeric column
customer_transaction_data[['amount_src', 'amount_usd', 'fee', 'device_trust_score', 'txn_velocity_1h', 'txn_velocity_24h', 'risk_score_internal']].lt(0).sum()

amount_src             0
amount_usd             0
fee                    0
device_trust_score     0
txn_velocity_1h        0
txn_velocity_24h       0
risk_score_internal    0
dtype: int64

In [42]:
# checking for any transaction timestamped in the future
customer_transaction_data[customer_transaction_data['timestamp'] > pd.Timestamp.utcnow()]

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud


In [43]:
# confirming the final row count, resolving whether any duplicate and missing-timestamp rows overlapped
customer_transaction_data.shape

(11140, 26)

In [44]:
# confirming location_mismatch has zero missing values
customer_transaction_data['location_mismatch'].value_counts(dropna=False)

location_mismatch
False    9322
True     1818
Name: count, dtype: int64

In [45]:
# checking whether an unknown channel correlates with fraud before deciding to treat it as missing
customer_transaction_data[customer_transaction_data['channel'] == 'unknown']['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.972973
1    0.027027
Name: proportion, dtype: float64

In [46]:
# converting the 'unknown' channel category to missing, since it doesn't concentrate fraud risk
customer_transaction_data['channel'] = customer_transaction_data['channel'].replace({'unknown': np.nan})
customer_transaction_data['channel'].unique()

array(['atm', 'web', 'mobile', nan], dtype=object)

In [47]:
# filling the newly-missing channel values with the most frequent channel
customer_transaction_data['channel'] = customer_transaction_data['channel'].fillna(
    customer_transaction_data['channel'].mode()[0]
)

In [48]:
# confirming channel has zero missing values after the fill
customer_transaction_data['channel'].isnull().sum()

np.int64(0)

In [49]:
# checking exact counts per kyc_tier variant before cleaning, and confirming no NaN slipped past the earlier fill
customer_transaction_data['kyc_tier'].value_counts(dropna=False)

kyc_tier
standard       8030
enhanced       1785
low            1030
standrd          72
STANDARD         69
 standard        63
unknown          30
 enhanced        17
ENHANCED         16
enhancd          12
 low              8
LOW               5
 nan              2
NAN               1
Name: count, dtype: int64

In [50]:
# standardizing kyc_tier formatting: trim whitespace and lowercase, mirroring the channel fix
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].str.strip().str.lower()

# correcting genuine misspellings the strip/lower fix can't catch
kyc_tier_corrections = {
    'standrd': 'standard',
    'enhancd': 'enhanced',
}
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].replace(kyc_tier_corrections)

# converting the literal text "nan" (a disguised missing value, not real NaN) back into an actual missing value
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].replace('nan', np.nan)

customer_transaction_data['kyc_tier'].value_counts(dropna=False)

kyc_tier
standard    8234
enhanced    1830
low         1043
unknown       30
NaN            3
Name: count, dtype: int64

In [51]:
# filling the 3 disguised-nan rows now revealed as real missing values, same mode-fill logic as the original imputation
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].fillna(
    customer_transaction_data['kyc_tier'].mode()[0]
)

In [52]:
# checking whether an unknown kyc_tier correlates with fraud before deciding to treat it as missing
customer_transaction_data[customer_transaction_data['kyc_tier'] == 'unknown']['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.966667
1    0.033333
Name: proportion, dtype: float64

In [53]:
# converting the 'unknown' kyc_tier category to missing, since it doesn't concentrate fraud risk
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].replace({'unknown': np.nan})

# filling with the mode, same logic used throughout
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].fillna(
    customer_transaction_data['kyc_tier'].mode()[0]
)

# confirming kyc_tier is fully clean
customer_transaction_data['kyc_tier'].value_counts(dropna=False)

kyc_tier
standard    8267
enhanced    1830
low         1043
Name: count, dtype: int64

In [54]:
# reviewing the home_country variable for formatting inconsistencies
customer_transaction_data['home_country'].unique()

array(['US', 'CA', 'UK', ' UK  ', ' US  ', 'unknown', ' CA  '],
      dtype=object)

In [55]:
# standardizing home_country formatting: trim whitespace and lowercase
customer_transaction_data['home_country'] = customer_transaction_data['home_country'].str.strip().str.lower()
customer_transaction_data['home_country'].unique()

array(['us', 'ca', 'uk', 'unknown'], dtype=object)

In [56]:
# checking whether an unknown home_country correlates with fraud before deciding to treat it as missing
customer_transaction_data[customer_transaction_data['home_country'] == 'unknown']['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.96875
1    0.03125
Name: proportion, dtype: float64

In [57]:
# converting the 'unknown' home_country category to missing, since it doesn't concentrate fraud risk
customer_transaction_data['home_country'] = customer_transaction_data['home_country'].replace({'unknown': np.nan})

# filling with the mode, same logic used throughout
customer_transaction_data['home_country'] = customer_transaction_data['home_country'].fillna(
    customer_transaction_data['home_country'].mode()[0]
)

# confirming home_country is fully clean
customer_transaction_data['home_country'].value_counts(dropna=False)

home_country
us    7842
uk    2092
ca    1206
Name: count, dtype: int64

In [58]:
# standardizing ip_country formatting: trim whitespace and lowercase, keeping "unknown" as its own meaningful category
customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].str.strip().str.lower()
customer_transaction_data['ip_country'].unique()

array(['us', 'ca', 'uk', 'unknown', 'nan'], dtype=object)

In [59]:
# checking how many rows carry the disguised "nan" text in ip_country
(customer_transaction_data['ip_country'] == 'nan').sum()

np.int64(3)

In [60]:
# converting the disguised "nan" text back into a real missing value
customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].replace('nan', np.nan)

# reapplying the same conditional fill used originally: home_country where no mismatch was flagged
mask_no_mismatch = customer_transaction_data['ip_country'].isnull() & (customer_transaction_data['location_mismatch'] == False)
customer_transaction_data.loc[mask_no_mismatch, 'ip_country'] = customer_transaction_data.loc[mask_no_mismatch, 'home_country']

# any row still missing had a confirmed mismatch, so it gets the same distinct placeholder as before
customer_transaction_data['ip_country'] = customer_transaction_data['ip_country'].fillna('unknown')

customer_transaction_data['ip_country'].isnull().sum()

np.int64(0)

## Why a Blanket dropna() Was Not Used

At this point the class walkthrough runs `customer_transaction_data.dropna(inplace=True)` on the full dataframe, dropping every row that has a missing value in any column at once.

This step was intentionally skipped. By this stage, every column with real missing values (`amount_usd`, `fee`, `kyc_tier`, `device_trust_score`, `ip_country`, `ip_address`, `channel`, `home_country`) had already been filled deliberately, using a method suited to that specific column, rather than discarded. Dropping rows here would remove real observations for no reason, and given the dataset is only 8.75% fraud, an unexamined drop risks disproportionately losing fraud cases, the exact failure mode already avoided earlier when `ip_address` was filled instead of dropped. A full-dataframe `dropna()` at this stage would not clean anything that hasn't already been cleaned, it would only discard data.

In [61]:
# scanning every text column for disguised missing values (nan-like strings pandas didn't recognize as real NaN)
object_cols = customer_transaction_data.select_dtypes(include='object').columns
for col in object_cols:
    suspicious = customer_transaction_data[col].astype(str).str.strip().str.lower().eq('nan')
    if suspicious.sum() > 0:
        print(col, suspicious.sum())

## Data Cleaning Complete

The dataset is now fully cleaned: zero missing values across all 26 columns, zero duplicate transaction records, zero invalid negative values in monetary, trust-score, and velocity columns, and consistent formatting across every categorical column reviewed (`channel`, `kyc_tier`, `home_country`, `ip_country`). `timestamp` and `amount_src` are correctly typed. The dataset is ready for exploratory data analysis and feature engineering.